In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D

In [ ]:
INPUT_ROOT  = "/Users/parth/Library/CloudStorage/Box-Box/WHT Datasets/01_columns_synced/CAMARGO"
OUTPUT_ROOT = "/Users/parth/Downloads/CAMARGO"

In [ ]:
# ==========================================================
# HELPER FUNCTIONS (defined once, outside the loop)
# ==========================================================
G0 = 9.80665

COL_PATTERNS = dict(
    acc=("{p}_ACC_X", "{p}_ACC_Y", "{p}_ACC_Z"),
    gyr=("{p}_GYR_X", "{p}_GYR_Y", "{p}_GYR_Z"),
)

def build_sensors_from_prefixes(df, prefixes, patterns):
    cols = set(df.columns)
    sensors = {}
    for p in prefixes:
        acc_cols = tuple(s.format(p=p) for s in patterns["acc"])
        if not all(c in cols for c in acc_cols):
            continue
        gyr_cols = tuple(s.format(p=p) for s in patterns.get("gyr", ()))
        gyr_ok = len(gyr_cols) == 3 and all(c in cols for c in gyr_cols)
        sensors[p] = {"acc": acc_cols, "gyr": gyr_cols if gyr_ok else None}
    return sensors

def find_static_window(acc, gyr=None, win=300, step=50):
    acc = np.asarray(acc, float)
    N = len(acc)
    if N <= win:
        return 0, N
    def acc_stability(a):
        return float(np.mean(np.std(a, axis=0)))
    if gyr is not None:
        gyr = np.asarray(gyr, float)
        gyr_mag = np.linalg.norm(gyr, axis=1)
        def score(s, e):
            return float(np.mean(gyr_mag[s:e]) + 0.5 * acc_stability(acc[s:e]))
    else:
        def score(s, e):
            return float(acc_stability(acc[s:e]))
    best = (np.inf, 0, win)
    for s in range(0, N - win + 1, step):
        e = s + win
        sc = score(s, e)
        if sc < best[0]:
            best = (sc, s, e)
    return best[1], best[2]

def detect_accel_units_and_scale(acc_static):
    acc_static = np.asarray(acc_static, float)
    mag = np.linalg.norm(acc_static, axis=1)
    med = float(np.median(mag))
    if 0.3 <= med <= 2.5:
        return {"unit_label": "g", "scale_to_g": 1.0, "median_mag": med}
    if 6.0 <= med <= 14.0:
        return {"unit_label": "m/s^2", "scale_to_g": 1.0 / G0, "median_mag": med}
    counts_per_g = med
    scale = 1.0 / counts_per_g
    return {"unit_label": "raw", "scale_to_g": scale, "median_mag": med, "counts_per_g_est": counts_per_g}

def normalize(v, eps=1e-12):
    v = np.asarray(v, float)
    n = np.linalg.norm(v)
    if n < eps:
        raise ValueError("Cannot normalize near-zero vector")
    return v / n

def rot_between(a, b):
    a = normalize(a); b = normalize(b)
    v = np.cross(a, b)
    c = float(np.clip(np.dot(a, b), -1.0, 1.0))
    s = np.linalg.norm(v)
    if s < 1e-12:
        if c > 0:
            return np.eye(3)
        tmp = np.array([1, 0, 0]) if abs(a[0]) < 0.9 else np.array([0, 1, 0])
        axis = normalize(np.cross(a, tmp))
        K = np.array([[0, -axis[2], axis[1]],
                      [axis[2], 0, -axis[0]],
                      [-axis[1], axis[0], 0]])
        return np.eye(3) + 2 * (K @ K)
    K = np.array([[0, -v[2], v[1]],
                  [v[2], 0, -v[0]],
                  [-v[1], v[0], 0]])
    R = np.eye(3) + K + (K @ K) * ((1 - c) / (s ** 2))
    return R

def rotate_series(R, V):
    V = np.asarray(V, float)
    return (R @ V.T).T

def angle_deg(u, v):
    u = normalize(u); v = normalize(v)
    c = float(np.clip(np.dot(u, v), -1.0, 1.0))
    return float(np.degrees(np.arccos(c)))

def rot_y(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def is_foot(sensor_name):
    return "FOOT" in sensor_name.upper()

def seg_height(name):
    n = name.upper()
    if 'FOOT'  in n: return -1.2
    if 'SHANK' in n: return  0.0
    if 'THIGH' in n: return  1.2
    return 0.0

def seg_side(name):
    return -0.6 if name.upper().startswith('L_') else 0.6

R_body = np.array([[ 0,  0, -1],
                   [ 1,  0,  0],
                   [ 0, -1,  0]], dtype=float)

R_foot = np.array([[ 0,  0, -1],
                   [ 1,  0,  0],
                   [ 0, -1,  0]], dtype=float)

R_ft2isb = np.array([[0, -1, 0],
                     [1,  0, 0],
                     [0,  0, 1]], dtype=float)

ROTATE_GYRO = True
STATIC_WIN  = 300
STATIC_STEP = 50
ISB_DOWN    = np.array([0., -1., 0.])

In [ ]:
# ==========================================================
# MAIN LOOP — process every CSV
# ==========================================================
csv_files = glob.glob(os.path.join(INPUT_ROOT, "**", "*.csv"), recursive=True)
print(f"Found {len(csv_files)} CSV files\n")

for csv_path in csv_files:
    print(f"Processing: {csv_path}")

    # ── Load ────────────────────────────────────────────────
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # ── Auto-detect sensors ─────────────────────────────────
    prefixes = sorted({c.replace("_ACC_X", "") for c in df.columns if c.endswith("_ACC_X")})
    SENSORS  = build_sensors_from_prefixes(df, prefixes, COL_PATTERNS)

    if not SENSORS:
        print(f"  ⚠ No sensors detected, skipping.\n")
        continue

    # ── Prepare output df ───────────────────────────────────
    df_out    = df.copy()
    report    = []
    rotations = {}

    for _s, _m in SENSORS.items():
        for _col in _m["acc"]:
            df_out[_col] = df_out[_col].astype("float64")
        if ROTATE_GYRO and _m.get("gyr") is not None:
            for _col in _m["gyr"]:
                df_out[_col] = df_out[_col].astype("float64")

    # ── Rotate each sensor ──────────────────────────────────
    for s, m in SENSORS.items():
        acc_cols = m["acc"]
        gyr_cols = m["gyr"]

        acc_raw = df.loc[:, acc_cols].to_numpy(float)
        gyr_raw = df.loc[:, gyr_cols].to_numpy(float) if gyr_cols is not None else None

        ws, we = find_static_window(acc_raw, gyr=gyr_raw,
                                    win=min(STATIC_WIN, len(df)), step=STATIC_STEP)
        info  = detect_accel_units_and_scale(acc_raw[ws:we])
        scale = float(info["scale_to_g"])
        acc_g = acc_raw * scale

        R = R_foot if is_foot(s) else R_body
        rotations[s] = R

        acc_isb = rotate_series(R, acc_g)
        df_out.loc[:, acc_cols[0]] = acc_isb[:, 0]
        df_out.loc[:, acc_cols[1]] = acc_isb[:, 1]
        df_out.loc[:, acc_cols[2]] = acc_isb[:, 2]

        if ROTATE_GYRO and gyr_raw is not None:
            gyr_isb = rotate_series(R, gyr_raw)
            df_out.loc[:, gyr_cols[0]] = gyr_isb[:, 0]
            df_out.loc[:, gyr_cols[1]] = gyr_isb[:, 1]
            df_out.loc[:, gyr_cols[2]] = gyr_isb[:, 2]

        g_after   = np.mean(acc_isb[ws:we], axis=0)
        ang_after = angle_deg(g_after, ISB_DOWN)

        report.append({
            "sensor":                  s,
            "static_start":            ws,
            "static_end":              we,
            "detected_unit":           info["unit_label"],
            "scale_to_g":              scale,
            "gravity_angle_after_deg": ang_after,
        })

    report_df = pd.DataFrame(report).sort_values("sensor")

    # ── Quick check print ───────────────────────────────────
    for row in report:
        s = row["sensor"]
        ws, we = int(row["static_start"]), int(row["static_end"])
        ax, ay, az = SENSORS[s]["acc"]
        gmean = df_out.loc[ws:we-1, [ax, ay, az]].to_numpy(float).mean(axis=0)
        print(f"  {s}: mean accel in static (g) [X,Y,Z] = {gmean}")

    # ── Plots ───────────────────────────────────────────────
    plt.rcParams.update({"figure.dpi": 300})
    colors   = ['red', 'green', 'blue']
    sensors  = report_df["sensor"].tolist()
    axes_lbl = ["X", "Y", "Z"]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 5), dpi=300)

    mean_before = np.zeros((len(sensors), 3))
    mean_after  = np.zeros((len(sensors), 3))
    for i, s in enumerate(sensors):
        row  = report_df[report_df["sensor"] == s].iloc[0]
        ws, we = int(row["static_start"]), int(row["static_end"])
        sc   = float(row["scale_to_g"])
        acols = SENSORS[s]["acc"]
        mean_before[i] = (df.loc[ws:we-1, acols].to_numpy(float) * sc).mean(axis=0)
        mean_after[i]  = df_out.loc[ws:we-1, acols].to_numpy(float).mean(axis=0)

    x = np.arange(len(sensors))
    w = 0.25

    for j, (aname, jcol) in enumerate(zip(axes_lbl, [0, 1, 2])):
        ax1.bar(x + (j-1)*w, mean_before[:, jcol], w, label=aname, alpha=0.85, color=colors[j])
    ax1.axhline(0,  color="gray",  linewidth=0.5)
    ax1.axhline(-1, color="green", linestyle="--", linewidth=0.7, alpha=0.7)
    ax1.set_xticks(x); ax1.set_xticklabels(sensors, rotation=45, ha="right")
    ax1.set_ylabel("Mean accel (g)"); ax1.set_title("Before (sensor frame, g)")
    ax1.legend(loc="upper right", fontsize=8); ax1.set_ylim(-1.5, 1.5)

    for j, (aname, jcol) in enumerate(zip(axes_lbl, [0, 1, 2])):
        ax2.bar(x + (j-1)*w, mean_after[:, jcol], w, label=aname, alpha=0.85, color=colors[j])
    ax2.axhline(0,  color="gray",  linewidth=0.5)
    ax2.axhline(-1, color="green", linestyle="--", linewidth=0.7, alpha=0.7)
    ax2.set_xticks(x); ax2.set_xticklabels(sensors, rotation=45, ha="right")
    ax2.set_ylabel("Mean accel (g)"); ax2.set_title("After (ISB); Y ≈ -1 g")
    ax2.legend(loc="upper right", fontsize=8); ax2.set_ylim(-1.5, 1.5)

    eg_s   = sensors[0]
    row    = report_df[report_df["sensor"] == eg_s].iloc[0]
    ws, we = int(row["static_start"]), int(row["static_end"])
    sc     = float(row["scale_to_g"])
    acols  = SENSORS[eg_s]["acc"]
    idx    = np.arange(ws, we)
    bef_y  = df.loc[ws:we-1, acols[1]].to_numpy(float) * sc
    aft_y  = df_out.loc[ws:we-1, acols[1]].to_numpy(float)
    ax3.plot(idx, bef_y, alpha=0.8, label="Before (sensor frame, g)", color="C0")
    ax3.plot(idx, aft_y, alpha=0.8, label="After (ISB)", color="C1")
    ax3.axhline(-1, color="green", linestyle="--", linewidth=0.8, alpha=0.8)
    ax3.set_xlabel("Sample index"); ax3.set_ylabel("Accel Y (g)")
    ax3.set_title(f"{eg_s} ACC_Y — after ≈ -1 g")
    ax3.legend(loc="upper right"); ax3.set_ylim(-1.5, 0.5)
    plt.tight_layout(); plt.show()

    # 3D plot
    sensor_list = list(rotations.keys())
    positions   = np.array([[seg_side(s), seg_height(s), 0.0] for s in sensor_list])
    arrow_scale = 0.4
    axis_colors = ["#e74c3c", "#27ae60", "#3498db"]
    axis_names  = ["X", "Y", "Z"]

    fig2 = plt.figure(figsize=(15, 5), dpi=300)
    ax_b = fig2.add_subplot(121, projection="3d")
    for i, s in enumerate(sensor_list):
        R_viz = (R_ft2isb @ rotations[s]) if is_foot(s) else rotations[s]
        px, py, pz = positions[i]
        for j in range(3):
            dx, dy, dz = arrow_scale * R_viz[:, j]
            ax_b.quiver(px, pz, py, dx, dz, dy, color=axis_colors[j], arrow_length_ratio=0.15, linewidth=1.5)
        ax_b.text(px+0.05, pz, py+0.05, s, fontsize=8)
    ax_b.set_xlabel("HuGaDB Z (backward)"); ax_b.set_ylabel("HuGaDB Y (right)"); ax_b.set_zlabel("HuGaDB X (up)")
    ax_b.set_title("Before"); ax_b.set_xlim(-2,2); ax_b.set_ylim(-2,2); ax_b.set_zlim(-2,2)
    ax_b.set_box_aspect([1,1,1]); ax_b.view_init(elev=20, azim=-60)
    leg = [Line2D([0],[0], color=axis_colors[j], lw=2, label=axis_names[j]) for j in range(3)]
    ax_b.legend(handles=leg, loc="upper left", frameon=False)

    ax_a = fig2.add_subplot(122, projection="3d")
    for i, s in enumerate(sensor_list):
        R_viz = R_ft2isb if is_foot(s) else np.eye(3)
        px, py, pz = positions[i]
        for j in range(3):
            dx, dy, dz = arrow_scale * R_viz[:, j]
            ax_a.quiver(px, pz, py, dx, dz, dy, color=axis_colors[j], arrow_length_ratio=0.15, linewidth=1.5)
        ax_a.text(px+0.05, pz, py+0.05, s, fontsize=8)
    ax_a.set_xlabel("ISB X (front)"); ax_a.set_ylabel("ISB Z (right)"); ax_a.set_zlabel("ISB Y (up)")
    ax_a.set_title("After"); ax_a.set_xlim(-2,2); ax_a.set_ylim(-2,2); ax_a.set_zlim(-2,2)
    ax_a.set_box_aspect([1,1,1]); ax_a.view_init(elev=20, azim=-60)
    leg = [Line2D([0],[0], color=axis_colors[j], lw=2, label=axis_names[j]) for j in range(3)]
    ax_a.legend(handles=leg, loc="upper left", frameon=False)
    plt.tight_layout(); plt.show()

    # ── Save — mirror input folder structure ────────────────
    rel_path = os.path.relpath(csv_path, INPUT_ROOT)
    out_path = os.path.join(OUTPUT_ROOT, rel_path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df_out.to_csv(out_path, index=False)
    print(f"  ✔ Saved -> {out_path}\n")

print("All done!")